# การสร้างแอปพลิเคชันสร้างภาพ (OpenAI)

มีมากกว่าการสร้างข้อความด้วย LLMs คุณยังสามารถสร้างภาพจากคำอธิบายข้อความได้ ภาพในฐานะรูปแบบข้อมูลมีประโยชน์ในหลายด้าน เช่น MedTech สถาปัตยกรรม การท่องเที่ยว การพัฒนาเกม การตลาด และอื่นๆ ในบทเรียนนี้เราจะทำงานกับโมเดล **GPT Image** ในแพลตฟอร์มของ OpenAI

## เป้าหมายการเรียนรู้

เมื่อจบบทเรียนนี้คุณจะสามารถ:

- อธิบายว่า การสร้างภาพคืออะไรและมีประโยชน์อย่างไร
- เข้าใจตระกูลโมเดล `gpt-image` และความแตกต่างจากโมเดล DALL·E รุ่นเก่า
- สร้างแอปพลิเคชันสร้างภาพและแก้ไขภาพได้

## การสร้างภาพคืออะไร?

โมเดลสร้างภาพจะสร้างภาพจากคำสั่งข้อความ โมเดลสมัยใหม่อย่าง `gpt-image` จะเรียนรู้ความสัมพันธ์ระหว่างข้อความกับภาพในระหว่างการฝึก จากนั้นจะแปลงสัญญาณรบกวนสุ่มให้เป็นภาพที่ตรงกับคำสั่งของคุณทีละขั้นตอน

ตระกูลโมเดลภาพที่เป็นที่รู้จักกันดีสองตระกูลคือ:

- **`gpt-image` (OpenAI)** — รุ่นปัจจุบันที่ใช้ในบทเรียนนี้ รองรับการสร้างภาพจากข้อความและการแก้ไขภาพ (วาดต่อเติมโดยใช้หน้ากาก).
- **Midjourney** — โมเดลของบุคคลที่สามยอดนิยมที่มีบริการและเวิร์กโฟลว์ผ่าน Discord ของตัวเอง.

> โมเดลภาพเก่าของ OpenAI — **DALL·E 2** และ **DALL·E 3** — เป็นรุ่นเก่า DALL·E 3 ไม่สามารถใช้งานสำหรับการติดตั้งใหม่ได้แล้ว และฟีเจอร์ต่างๆ เช่น `create_variation` มีเพียงใน DALL·E 2 เท่านั้น ใช้โมเดล `gpt-image` สำหรับแอปพลิเคชันใหม่

> **สำคัญ:** โมเดล `gpt-image` จะส่งคืนภาพที่สร้างเป็น **base64** (`b64_json`) ไม่ใช่ URL รหัสของคุณจะถอดรหัสสตริง base64 เป็นไบต์และบันทึก — ไม่มี URL ของภาพให้ดาวน์โหลด


## การสร้างแอปพลิเคชันการสร้างภาพแรกของคุณ

แล้วต้องใช้อะไรบ้างในการสร้างแอปพลิเคชันการสร้างภาพ? คุณจำเป็นต้องมีไลบรารีต่อไปนี้:

- **python-dotenv**, เราขอแนะนำอย่างยิ่งให้ใช้ไลบรารีนี้เพื่อเก็บความลับของคุณในไฟล์ *.env* แยกจากโค้ด
- **openai**, ไลบรารีนี้คือสิ่งที่คุณจะใช้ในการติดต่อกับ OpenAI API
- **pillow**, เพื่อทำงานกับภาพใน Python


1. สร้างไฟล์ *.env* ด้วยเนื้อหาต่อไปนี้:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. รวบรวมไลบรารีด้านบนในไฟล์ที่ชื่อว่า *requirements.txt* ดังนี้:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. ต่อไป ให้สร้าง environment เสมือนและติดตั้งไลบรารี:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> สำหรับ Windows ให้ใช้คำสั่งต่อไปนี้เพื่อสร้างและเปิดใช้งานสภาพแวดล้อมเสมือนของคุณ:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. เพิ่มโค้ดต่อไปนี้ในไฟล์ที่ชื่อ *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

มาอธิบายโค้ดนี้กัน:

- ก่อนอื่น เรานำเข้าห้องสมุดที่จำเป็นทั้งหมด รวมถึงห้องสมุด OpenAI, ห้องสมุด dotenv, โมดูล base64 และห้องสมุด Pillow

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- หลังจากนั้น เราสร้างลูกค้า client ซึ่งจะอ่านคีย์ API จาก ``.env`` ของคุณ

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- ต่อไป เราสร้างภาพ:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` models จะคืนภาพในรูปแบบสตริง **base64** ใน `data[0].b64_json` เราจะถอดรหัสเป็นไบต์และเขียนลงไฟล์ — ไม่มี URL ให้ดาวน์โหลด

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- สุดท้าย เราเปิดภาพและใช้โปรแกรมดูภาพมาตรฐานเพื่อแสดงภาพนั้น:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### รายละเอียดเพิ่มเติมเกี่ยวกับการสร้างภาพ

มาดูพารามิเตอร์ของ `images.generate` กัน:

- **model**, คือโมเดลภาพที่ใช้ (ตัวอย่างเช่น `gpt-image-2`).
- **prompt**, คือข้อความคำสั่งที่ใช้สร้างภาพ ที่นี่คือ "กระต่ายบนหลังม้า ถืออมยิ้ม อยู่บนทุ่งหมอกที่มีดอกแดฟโฟดิลขึ้น".
- **size**, คือขนาดของภาพที่สร้างขึ้น (ตัวอย่างเช่น `1024x1024`, `1536x1024`, `1024x1536`, หรือ `"auto"`).
- **n**, คือจำนวนภาพที่สร้างขึ้น ที่นี่เราสร้างหนึ่งภาพ.

> โมเดลภาพไม่มีพารามิเตอร์ `temperature` — นั่นคือการควบคุมการสร้างข้อความ หากต้องการความหลากหลาย ให้เรียก API อีกครั้ง; หากต้องการลดความหลากหลาย ให้ทำพรอมต์ของคุณให้เฉพาะเจาะจงขึ้น

## ความสามารถเพิ่มเติมของการสร้างภาพ

คุณได้เห็นวิธีสร้างภาพด้วยโค้ด Python ไม่กี่บรรทัดแล้ว `gpt-image` โมเดลยังสามารถ **แก้ไข** ภาพที่มีอยู่ได้ โดยการให้ภาพ, **หน้ากาก** (optional) ซึ่งเป็นการทำเครื่องหมายพื้นที่ที่ต้องการเปลี่ยนแปลง และพรอมต์ คุณสามารถเปลี่ยนส่วนหนึ่งของภาพ — เช่น เพิ่มหมวกให้กระต่ายของเรา

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

ภาพพื้นฐานมีแค่กระต่ายเท่านั้น; ภาพสุดท้ายมีหมวกบนหัวกระต่าย


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
